<a href="https://colab.research.google.com/github/adljna/ProjectA-Group3-KematianAliKhamenei/blob/main/POS%20Tagging/Detik/3-Detik-POSTagging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **POS Tagging Berita Kematian Ali Khamenei - Detik**

Notebook ini berisi proses Part-of-Speech (POS) Tagging pada data artikel yang telah diproses sebelumnya. Langkah-langkah utama yang dilakukan dalam notebook ini meliputi:

*   **Instalasi & Setup**: Menginstal dan mengimpor pustaka Python yang diperlukan seperti `pandas`, `nltk`, dan `stanza` untuk pemrosesan bahasa alami (NLP) Indonesia.
*   **Load Data**: Mengunggah dan memuat file CSV (`analyzed_articles.csv`) yang berisi data artikel yang sudah melalui tahap pra-pemrosesan (seperti pembersihan teks, tokenisasi, penghapusan stopwords, stemming, dan analisis sentimen).
*   **POS Tagging**: Menerapkan POS tagging pada kolom `text_final` menggunakan model Stanza bahasa Indonesia untuk mengidentifikasi kategori gramatikal setiap kata (misalnya, kata benda, kata kerja, kata sifat).
*   **Analisis Frekuensi POS**: Menganalisis dan menampilkan frekuensi kata-kata teratas untuk setiap jenis POS yang relevan, memberikan wawasan tentang distribusi kata dalam teks.
*   **Ekspor Hasil**: Menyimpan DataFrame yang telah diperbarui, termasuk kolom `pos_tags` yang baru, ke dalam file CSV baru (`tagged_articles.csv`).

# **(1) Installation & Setup**

Ini adalah bagian untuk menginstal dan mengimpor semua pustaka Python yang diperlukan untuk analisis teks. Ini mencakup `pandas` untuk manipulasi data, `nltk` dan `stanza` untuk pemrosesan bahasa alami (NLP) Indonesia, `tqdm` untuk progress bar, `io` untuk input/output data, dan `google.colab.files` untuk mengunggah file. Model bahasa Indonesia untuk NLTK dan Stanza juga diunduh dan diinisialisasi di sini.

In [ ]:
!pip install nlp-id transformers torch tqdm stanza

In [ ]:
from google.colab import files
import io
import pandas as pd
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
import stanza
from collections import Counter, defaultdict
from tqdm import tqdm

tqdm.pandas()

# **(2) Load Data**

Pada bagian ini, Anda akan mengunggah file CSV yang berisi data artikel. Kode akan membaca file yang diunggah ke dalam DataFrame `df_article` menggunakan pustaka `pandas` dan `io`.

In [ ]:
from google.colab import files
uploaded = files.upload()

import io
import pandas as pd

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

  # Assuming the uploaded file is a CSV
  try:
    df_article = pd.read_csv(io.BytesIO(uploaded[fn]))
    print("Successfully loaded CSV into df_article")
  except pd.errors.ParserError:
      print(f"Error: Could not parse {fn} as a CSV. Please upload a valid CSV file.")
      df_article = pd.DataFrame() # Create an empty DataFrame if parsing fails
  except Exception as e:
    print(f"An unexpected error occurred: {e}")
    df_article = pd.DataFrame()

Saving analyzed_articles.csv to analyzed_articles.csv
User uploaded file "analyzed_articles.csv" with length 1492921 bytes
Successfully loaded CSV into df_article


In [ ]:
print(df_article.columns)

Index(['url', 'title', 'content', 'text_cleaned', 'tokens', 'tokens_no_stop',
       'tokens_stemmed', 'text_final', 'polarity', 'subjectivity',
       'sentiment_label'],
      dtype='object')


In [ ]:
df_article

,url,title,content,text_cleaned,tokens,tokens_no_stop,tokens_stemmed,text_final,polarity,subjectivity,sentiment_label
0,https://news.detik.com/berita/d-8392835/puan-b...,Puan Berduka Ali Khamenei Gugur Diserang AS-Is...,Ketua DPR RI Puan Maharani menyampaikan dukaci...,ketua dpr ri puan maharani menyampaikan dukaci...,"['ketua', 'dpr', 'ri', 'puan', 'maharani', 'me...","['ketua', 'dpr', 'ri', 'puan', 'maharani', 'du...","['ketua', 'dpr', 'ri', 'puan', 'maharani', 'du...",ketua dpr ri puan maharani dukacita masyarakat...,0.013725,0.203922,Netral
1,https://news.detik.com/internasional/d-8392814...,Trump Beri Sinyal Perang Lawan Iran Akan Selesai,Presiden Amerika Serikat (AS) Donald Trump men...,presiden amerika serikat as donald trump menga...,"['presiden', 'amerika', 'serikat', 'as', 'dona...","['presiden', 'amerika', 'serikat', 'as', 'dona...","['presiden', 'amerika', 'serikat', 'as', 'dona...",presiden amerika serikat as donald trump peran...,0.060000,0.240000,Positive
2,https://www.detik.com/bali/berita/d-8392847/pe...,Peringatan Keras Iran ke Kapal Tanker yang Nek...,Peringatan Keras Iran ke Kapal Tanker yang Nek...,peringatan keras iran ke kapal tanker yang nek...,"['peringatan', 'keras', 'iran', 'ke', 'kapal',...","['peringatan', 'keras', 'iran', 'kapal', 'tank...","['ingat', 'keras', 'iran', 'kapal', 'tanker', ...",ingat keras iran kapal tanker nekat lewat sela...,0.115345,0.484360,Positive
3,https://finance.detik.com/energi/d-8392622/ira...,Iran Beri Peringatan Keras Kapal Tanker yang M...,"Juru Bicara Kementerian Luar Negeri Iran, Esma...",juru bicara kementerian luar negeri iran esmai...,"['juru', 'bicara', 'kementerian', 'luar', 'neg...","['juru', 'bicara', 'kementerian', 'negeri', 'i...","['juru', 'bicara', 'menteri', 'negeri', 'iran'...",juru bicara menteri negeri iran esmail baghaei...,0.004545,0.414773,Netral
4,https://finance.detik.com/energi/d-8392622/ira...,Iran Beri Peringatan Keras Kapal Tanker yang M...,"Juru Bicara Kementerian Luar Negeri Iran, Esma...",juru bicara kementerian luar negeri iran esmai...,"['juru', 'bicara', 'kementerian', 'luar', 'neg...","['juru', 'bicara', 'kementerian', 'negeri', 'i...","['juru', 'bicara', 'menteri', 'negeri', 'iran'...",juru bicara menteri negeri iran esmail baghaei...,0.004545,0.414773,Netral
...,...,...,...,...,...,...,...,...,...,...,...
95,https://www.detik.com/hikmah/khazanah/d-796858...,"Iran Yakin Menang Lawan Israel, Ali Khamenei K...",Iran tengah terlibat konflik bersenjata dengan...,iran tengah terlibat konflik bersenjata dengan...,"['iran', 'tengah', 'terlibat', 'konflik', 'ber...","['iran', 'terlibat', 'konflik', 'bersenjata', ...","['iran', 'libat', 'konflik', 'senjata', 'israe...",iran libat konflik senjata israel pimpin tingg...,0.193957,0.409864,Positive
96,https://www.detik.com/hikmah/khazanah/d-796858...,"Iran Yakin Menang Lawan Israel, Ali Khamenei K...",Iran tengah terlibat konflik bersenjata dengan...,iran tengah terlibat konflik bersenjata dengan...,"['iran', 'tengah', 'terlibat', 'konflik', 'ber...","['iran', 'terlibat', 'konflik', 'bersenjata', ...","['iran', 'libat', 'konflik', 'senjata', 'israe...",iran libat konflik senjata israel pimpin tingg...,0.193957,0.409864,Positive
97,https://www.detik.com/bali/berita/d-7965979/as...,AS Tolak Rencana Israel Bunuh Ali Khamenei,AS Tolak Rencana Israel Bunuh Ali Khamenei\n\n...,as tolak rencana israel bunuh ali khamenei far...,"['as', 'tolak', 'rencana', 'israel', 'bunuh', ...","['as', 'tolak', 'rencana', 'israel', 'bunuh', ...","['as', 'tolak', 'rencana', 'israel', 'bunuh', ...",as tolak rencana israel bunuh ali khamenei far...,0.133629,0.435809,Positive
98,https://news.detik.com/internasional/d-7965911...,Trump Larang Israel Bunuh Pemimpin Tertinggi I...,Presiden Amerika Serikat (AS) Donald Trump men...,presiden amerika serikat as donald trump menol...,"['presiden', 'amerika', 'serikat', 'as', 'dona...","['presiden', 'amerika', 'serikat', 'as', 'dona...","['presiden', 'amerika', 'serikat', 'as', 'dona...

# **(3) POS tagging**


Bagian ini menerapkan Part-of-Speech (POS) tagging pada kolom `text_final` dalam DataFrame `df_article`. Setiap kata dalam teks akan diberi label berdasarkan kelas katanya (misalnya, kata benda, kata kerja, kata sifat) menggunakan model `stanza` yang telah diinisialisasi. Hasilnya akan disimpan dalam kolom baru bernama `pos_tags`.

In [ ]:
# Download the Indonesian Stanza model if not already downloaded
stanza.download('id')

# Initialize the Stanza pipeline for tokenization and POS tagging
stanza_nlp = stanza.Pipeline('id', processors='tokenize,pos')

# Create wrapper class to mimic the interface of the original pos_tagger
class StanzaPosTagger:
    def get_pos_tag(self, text):
        doc = stanza_nlp(text)
        pos_tags = []
        for sent in doc.sentences:
            for word in sent.words:
                pos_tags.append((word.text, word.pos))
        return pos_tags

pos_tagger = StanzaPosTagger()

INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: id (Indonesian) ...
INFO:stanza:File exists: /root/.cache/stanza/1.11.0/resources/id/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: id (Indonesian):
| Processor | Package    |
--------------------------
| tokenize  | gsd        |
| mwt       | gsd        |
| pos       | gsd_charlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Done loading processors!


In [ ]:
# Make sure the text column is string
df_article["text_final"] = df_article["text_final"].astype(str)

# Apply POS tagging
df_article["pos_tags"] = df_article["text_final"].apply(lambda x: pos_tagger.get_pos_tag(x))

In [ ]:
# Flatten the list of lists of (word, tag) tuples into a single list of dictionaries
word_pos_list = []
for index, row in df_article.iterrows():
    if isinstance(row['pos_tags'], list):
        for word, pos_tag in row['pos_tags']:
            word_pos_list.append({'Word': word, 'POS Tag': pos_tag})

df_word_pos = pd.DataFrame(word_pos_list)
print("\nDataFrame Kata dan POS Tag dari Stanza:")
display(df_word_pos.head())

# Menghitung frekuensi tag dan kata unik per tag
tag_counts = df_word_pos['POS Tag'].value_counts()
unique_tokens_per_tag = df_word_pos.groupby('POS Tag')['Word'].nunique()

# Create summary DataFrame, ensuring index alignment for unique_tokens_per_tag
summary_df = pd.DataFrame({
    'Tag': tag_counts.index,
    'Count': tag_counts.values,
    'Unique Tokens': unique_tokens_per_tag.reindex(tag_counts.index)
}).reset_index(drop=True)
summary_df = summary_df.sort_values(by='Count', ascending=False)

print("\nRingkasan POS Tagging:")
display(summary_df)
print(f"Total kata yang di-tag: {summary_df['Count'].sum()}")

# Save summary to CSV
summary_df.to_csv("pos_tags_summary.csv", index=False)


DataFrame Kata dan POS Tag dari Stanza:


,Word,POS Tag
0,ketua,NOUN
1,dpr,NOUN
2,ri,PROPN
3,puan,NOUN
4,maharani,NOUN



Ringkasan POS Tagging:


,Tag,Count,Unique Tokens
0,NOUN,17448,1997
1,ADJ,1517,173
2,PROPN,1040,120
3,VERB,795,111
4,X,793,220
5,PART,190,2
6,ADP,106,25
7,NUM,72,17
8,ADV,51,14
9,PRON,45,7


Total kata yang di-tag: 22075


In [ ]:
df_article[["text_final", "pos_tags"]].head(10)

,text_final,pos_tags
0,ketua dpr ri puan maharani dukacita masyarakat...,"[(ketua, NOUN), (dpr, NOUN), (ri, PROPN), (pua..."
1,presiden amerika serikat as donald trump peran...,"[(presiden, NOUN), (amerika, PROPN), (serikat,..."
2,ingat keras iran kapal tanker nekat lewat sela...,"[(ingat, VERB), (keras, ADJ), (iran, NOUN), (k..."
3,juru bicara menteri negeri iran esmail baghaei...,"[(juru, NOUN), (bicara, VERB), (menteri, NOUN)..."
4,juru bicara menteri negeri iran esmail baghaei...,"[(juru, NOUN), (bicara, VERB), (menteri, NOUN)..."
5,respons trump australia tampung main timnas pu...,"[(respons, NOUN), (trump, PROPN), (australia, ..."
6,australia tampung main timnas putri iran kabur...,"[(australia, PROPN), (tampung, VERB), (main, N..."
7,perang amerika serikat as israel iran harga mi...,"[(perang, NOUN), (amerika, PROPN), (serikat, N..."
8,donald trump sorot nasib timnas putri iran sin...,"[(donald, PROPN), (trump, PROPN), (sorot, NOUN..."
9,tekateki sosok pimpin tinggi iran mati ayatoll...,"[(tekateki, NOUN), (sosok, NOUN), (pimpin, NOU..."


# (4) POS Frequency Analysis

Kode ini menganalisis frekuensi kemunculan kata-kata berdasarkan tag POS-nya. Ini akan menghitung 5 kata teratas untuk setiap jenis POS yang relevan (seperti kata benda, kata kerja, kata sifat, dll.), memberikan gambaran tentang kata-kata yang paling sering digunakan dalam setiap kategori.

In [ ]:
from collections import defaultdict, Counter

# --- Analisis Frekuensi POS (Part-of-Speech) ---

# Buat dictionary untuk menyimpan Counter untuk setiap tag POS
pos_counts = defaultdict(Counter)

# Iterasi setiap baris di DataFrame
# df_article['pos_tags'] berisi list of tuples, misal: [('lion', 'NN'), ('air', 'NN')]
for list_of_tags in df_article['pos_tags']:
    for word, tag in list_of_tags:
        pos_counts[tag][word] += 1

print("--- Frekuensi Top 5 Kata per Jenis POS ---")

# Tampilkan 5 kata paling umum untuk setiap tag POS yang relevan
# Stanza uses Universal Dependencies (UD) tags. Common ones include NOUN, VERB, ADJ, ADV, PROPN.
relevant_pos_tags = ['NOUN', 'VERB', 'ADJ', 'ADV', 'PROPN'] # Updated to use UD tags

for tag in relevant_pos_tags:
    if tag in pos_counts:
        print(f"\nTop 5 untuk tag '{tag}':")
        print(pos_counts[tag].most_common(5))


--- Frekuensi Top 5 Kata per Jenis POS ---

Top 5 untuk tag 'NOUN':
[('iran', 1026), ('khamenei', 613), ('pimpin', 438), ('israel', 310), ('serang', 292)]

Top 5 untuk tag 'VERB':
[('tewas', 95), ('sebut', 66), ('menang', 38), ('bunuh', 38), ('pilih', 36)]

Top 5 untuk tag 'ADJ':
[('tinggi', 229), ('militer', 112), ('aman', 79), ('kuat', 69), ('salah', 54)]

Top 5 untuk tag 'ADV':
[('tentu', 13), ('terus', 11), ('turut', 6), ('hendak', 6), ('kurang', 3)]

Top 5 untuk tag 'PROPN':
[('as', 215), ('amerika', 131), ('to', 74), ('serikat', 71), ('trump', 48)]


In [ ]:
df_article

,url,title,content,text_cleaned,tokens,tokens_no_stop,tokens_stemmed,text_final,polarity,subjectivity,sentiment_label,pos_tags
0,https://news.detik.com/berita/d-8392835/puan-b...,Puan Berduka Ali Khamenei Gugur Diserang AS-Is...,Ketua DPR RI Puan Maharani menyampaikan dukaci...,ketua dpr ri puan maharani menyampaikan dukaci...,"['ketua', 'dpr', 'ri', 'puan', 'maharani', 'me...","['ketua', 'dpr', 'ri', 'puan', 'maharani', 'du...","['ketua', 'dpr', 'ri', 'puan', 'maharani', 'du...",ketua dpr ri puan maharani dukacita masyarakat...,0.013725,0.203922,Netral,"[(ketua, NOUN), (dpr, NOUN), (ri, PROPN), (pua..."
1,https://news.detik.com/internasional/d-8392814...,Trump Beri Sinyal Perang Lawan Iran Akan Selesai,Presiden Amerika Serikat (AS) Donald Trump men...,presiden amerika serikat as donald trump menga...,"['presiden', 'amerika', 'serikat', 'as', 'dona...","['presiden', 'amerika', 'serikat', 'as', 'dona...","['presiden', 'amerika', 'serikat', 'as', 'dona...",presiden amerika serikat as donald trump peran...,0.060000,0.240000,Positive,"[(presiden, NOUN), (amerika, PROPN), (serikat,..."
2,https://www.detik.com/bali/berita/d-8392847/pe...,Peringatan Keras Iran ke Kapal Tanker yang Nek...,Peringatan Keras Iran ke Kapal Tanker yang Nek...,peringatan keras iran ke kapal tanker yang nek...,"['peringatan', 'keras', 'iran', 'ke', 'kapal',...","['peringatan', 'keras', 'iran', 'kapal', 'tank...","['ingat', 'keras', 'iran', 'kapal', 'tanker', ...",ingat keras iran kapal tanker nekat lewat sela...,0.115345,0.484360,Positive,"[(ingat, VERB), (keras, ADJ), (iran, NOUN), (k..."
3,https://finance.detik.com/energi/d-8392622/ira...,Iran Beri Peringatan Keras Kapal Tanker yang M...,"Juru Bicara Kementerian Luar Negeri Iran, Esma...",juru bicara kementerian luar negeri iran esmai...,"['juru', 'bicara', 'kementerian', 'luar', 'neg...","['juru', 'bicara', 'kementerian', 'negeri', 'i...","['juru', 'bicara', 'menteri', 'negeri', 'iran'...",juru bicara menteri negeri iran esmail baghaei...,0.004545,0.414773,Netral,"[(juru, NOUN), (bicara, VERB), (menteri, NOUN)..."
4,https://finance.detik.com/energi/d-8392622/ira...,Iran Beri Peringatan Keras Kapal Tanker yang M...,"Juru Bicara Kementerian Luar Negeri Iran, Esma...",juru bicara kementerian luar negeri iran esmai...,"['juru', 'bicara', 'kementerian', 'luar', 'neg...","['juru', 'bicara', 'kementerian', 'negeri', 'i...","['juru', 'bicara', 'menteri', 'negeri', 'iran'...",juru bicara menteri negeri iran esmail baghaei...,0.004545,0.414773,Netral,"[(juru, NOUN), (bicara, VERB), (menteri, NOUN)..."
...,...,...,...,...,...,...,...,...,...,...,...,...
95,https://www.detik.com/hikmah/khazanah/d-796858...,"Iran Yakin Menang Lawan Israel, Ali Khamenei K...",Iran tengah terlibat konflik bersenjata dengan...,iran tengah terlibat konflik bersenjata dengan...,"['iran', 'tengah', 'terlibat', 'konflik', 'ber...","['iran', 'terlibat', 'konflik', 'bersenjata', ...","['iran', 'libat', 'konflik', 'senjata', 'israe...",iran libat konflik senjata israel pimpin tingg...,0.193957,0.409864,Positive,"[(iran, NOUN), (libat, NOUN), (konflik, NOUN),..."
96,https://www.detik.com/hikmah/khazanah/d-796858...,"Iran Yakin Menang Lawan Israel, Ali Khamenei K...",Iran tengah terlibat konflik bersenjata dengan...,iran tengah terlibat konflik bersenjata dengan...,"['iran', 'tengah', 'terlibat', 'konflik', 'ber...","['iran', 'terlibat', 'konflik', 'bersenjata', ...","['iran', 'libat', 'konflik', 'senjata', 'israe...",iran libat konflik senjata israel pimpin tingg...,0.193957,0.409864,Positive,"[(iran, NOUN), (libat, NOUN), (konflik, NOUN),..."
97,https://www.detik.com/bali/berita/d-7965979/as...,AS Tolak Rencana Israel Bunuh Ali Khamenei,AS Tolak Rencana Israel Bunuh Ali Khamenei\n\n...,as tolak rencana israel bunuh ali khamenei far...,"['as', 'tolak', 'rencana', 'israel', 'bunuh', ...","['as', 'tolak', 'rencana', 'israel', 'bunuh', ...","['as', 'tolak', 'rencana', 'israel', 'bunuh', ...",as tolak rencana israel bunuh ali khamenei far...,0.13362

# (5) Save Processed Data

Setelah semua langkah pemrosesan selesai, DataFrame yang telah diperbarui dengan tag POS akan disimpan ke dalam file CSV baru dengan nama `data/tagged_articles.csv`. Parameter `index=False` memastikan bahwa indeks DataFrame tidak ikut disimpan ke dalam file CSV.

In [ ]:
# Save the DataFrame to a CSV file
df_article.to_csv('tagged_articles.csv', index=False)